In [0]:
spark.conf.get("spark.sql.files.maxPartitionBytes")

In [0]:
spark.conf.set("spark.sql.files.maxPartitionBytes", 134217728)
#set the max partition size to 10000 Bytes, now more partitions will be created to hold data

In [0]:
aapl_data_new = spark.sql('select * from aapl_data')

In [0]:
repartitioned_aapl_data = aapl_data_new.repartition(8)

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", False)
#this is required when manually repartitioning, otherwise spark will work as per max size in paritition

In [0]:
spark.conf.get("spark.sql.shuffle.partitions")

In [0]:
#df.rdd.glom().collect()
#gives the values of all partitions in a single list of lists (1 list for each partition)

In [0]:
coalesced_aapl_data = repartitioned_aapl_data.coalesce(2)
#reduce partitions, it cannot be used to increase the number of partitions

In [0]:
# df1_df2 = df1.union(df2)
#union of 2 dataframes

In [0]:
#df1_df2 = df1.join(df2,
#                    ['joincolumn'],
#                    how='inner')
#leftsemi join gives only those records from left dataframe which have a match in right dataframe
#leftanti join gives only those records from left dataframe which do not have a match in right daraframe


In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf, col, PandasUDFType
from pyspark.sql.types import IntegerType

def year(date: pd.Series) -> pd.Series:
    return(pd.to_datetime(date).dt.year)
#vectorized udf
year_pandas = pandas_udf(year, returnType=IntegerType())
aapl_data_year = aapl_data_new.withColumn('Year', year_pandas(col('date')))\
    .select('date', 'volume', 'open', 'Year')
aapl_data_year.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

window_spec = Window.partitionBy(aapl_data_year['Year'])\
                    .orderBy(aapl_data_year['volume'].desc())
volume_rank = rank().over(window_spec)

In [0]:
volume_rank

In [0]:
volume_rank_df = aapl_data_year.select('date', 'volume', 'open', 'Year')\
                                .withColumn('volume_rank', volume_rank)
volume_rank_df.display()

In [0]:
volume_rank_df.filter(volume_rank_df['volume_rank'] == 1).display()

In [0]:
from pyspark.sql.functions import max
window_spec = Window.partitionBy(aapl_data_year['Year'])\
                    .orderBy(aapl_data_year['volume'].desc())\
                    .rowsBetween(-1,0)
compare_volume = max(aapl_data_year['volume']).over(window_spec)


In [0]:
aapl_data_year.select('date', 'volume', 'open', 'Year')\
                .withColumn('comparevolume', compare_volume).display()
#now we see the volume of the next highest volume